# NTRU Digital Signature



#### The NTRU cryptosystem

- Step 1  \(<span style='color:blue'>public</span>\) Alice or Bob \(or some trusted third party\) chooses public parameters $(N, p, q, d)$ with $N$ and $p$ prime, $\gcd(p, q) = \gcd(N, q) = 1,$ $N > 2d+1$ and $q > (6d+1)p.$  Again, this data is public.  Alice, Bob and Eve all know it.
- Step 2  \(<span style='color:red'>private</span>\)  Alice chooses a private polynomial $f \in  \mathcal T (N,d + 1, d)$ that has inverses modulo $p$ and modulo $q$ with respect to convolution. She computes the polynomials $F_p$ and $F_q$ of degree $ < N$ such that $$f *F_p \equiv 1 \!\!\! \pmod p \quad \textrm{and} \quad f*F_q \equiv 1 \!\!\! \pmod q.$$
  She also chooses a private $g \in \mathcal T (N,d, d).$ The polynomials $(f,g,F_p, F_q)$ are her _private key._ Technically one can recover $F_p$ and $F_q$ from $f(x)$, but it is better to keep track of the inverses as well.
  Alice also computes $$h = F_q * g.$$
- Step 3  \(<span style='color:blue'>public</span>\) Alice publishes $h(x)$. This is Alice's _public key_ and Eve sees it.
- Step 4 \(<span style='color:red'>private</span>\)  The is the **encryption step**. Bob chooses as plaintext a polynomial $m$ of degree $ < N$ with integer coefficients modulo $p.$ One can either make the convention that the coefficients take values in $[0, p-1]$ or in $(-p/2,p/2)$ \(i.e. work with centered polynomials\).

Bob also  chooses as \(secret\) ephemeral key a polynomial $r \in \mathcal T(N,d,d).$ He uses Alice's public key $h$ to compute the ciphertext $$ c \equiv p (r*h) + m \pmod q.$$

- Step 5  \(<span style='color:blue'>public</span>\) Bob sends the ciphertext $c$ to Alice. Eve can see this.
- Step 6 \(<span style='color:red'>private</span>\) This is the **decryption step**. Alice computes first $f*c$ and its centerlift modulo $q,$ denoted by $b(x).$ Recall that we think of $b(x)$ as an ordinary polynomial with integer coefficients. Alice recovers Bob's message by computing $$m \equiv F_p * b \pmod p.$$



##### Why does this work? In other words, what makes Alice so sure she recovered Bob's plaintext $m$



First, we have $$b(x)\equiv f*c \equiv f * (p \cdot r*h + m) \equiv p \cdot f*r*h + f*m \equiv p \cdot f*r * F_q * g + f*m \pmod q.$$ But $$f*r * F_q * g = (f*F_q) * r* g \equiv r*g \pmod q.$$ Therefore $$b \equiv p \cdot  r*g + f*m\pmod q.$$

Since $f, g, m$ and $r$ have small coefficients and $p$ is much smaller than $q,$ it is very probable that $p(r*g) + f*m,$ before reducing modulo $q,$ has coefficients of absolute value less than $q/2.$ In this case, we have equality
$$ b =  p (r*g) + f*m. \qquad \qquad (1)$$
The whole reason we work with polynomials in $\mathcal T(N,d_1, d_2)$  and numbers $N> (6d+1)q$ is to guarantee that this happens.

The equality \(1\) implies $$ F_p * b =  p (F_p* r*g) + F_p * f*m  \equiv F_p *f * m \equiv m\pmod p.$$



In [1]:
R.<x>=PolynomialRing(ZZ)
N = 101
p = 3
d = 10
q = 6263
f = x^50 + x^48 + x^45 - x^40 - x^38 - x^35 + x^34 - x^32 + x^27 + x^25 + x^23 - x^18 - x^16 + x^14 + x^11 + x^9 - x^7 - x^6 - x^5 - x^2 + 1
g = x^52 + x^51 + x^50 + x^48 + x^45 - x^44 + x^43 + x^40 + x^39 - x^37 - x^35 + x^30 - x^29 - x^23 - x^17 - x^11 - x^7 - x^6 + x^2 - x

In [0]:
S = R.quotient(1-x^N)
Rp.<u> = PolynomialRing(GF(p))
Sp = Rp.quotient(1-x^N)

Rq.<u> = PolynomialRing(GF(q))
Sq = Rq.quotient(1-x^N)

In [0]:
Fpbar = Sp(f)^(-1)
Fp = 0
for i in range(N):
    Fp = Fp + ZZ(list(Fpbar)[i])*x^i

In [0]:
Fqbar = Sq(f)^(-1)
Fq = 0
for i in range(N):
    Fq = Fq + ZZ(list(Fqbar)[i])*x^i

In [0]:
hbar = S(Fp)*S(g)
h = 0
for i in range(N):
    h = h + ZZ(list(S(Fp)*S(g))[i])*x^i    

m = x^3\+x^5\+x^6 \+ x^8\*\(1\+x^2\+x^5\+x^6\) \+ x^16\*\(x^2\+x^3\+x^5\+x^6\) \+ x^24\*\(x^2\+x^3\+x^5\+x^6\)\+ x^32\*\(1\+x\+x^2\+x^3\+x^5\+x^6\)\+ x^40\*\(1\+x\+x^2\+x^4\+x^5\+x^6\) \+ x^48\*\(1\+x\+x^2\+x^3\+x^5\+x^6\) \+ x^56\*\(x\+x^4\+x^5\+x^6\) \+ x^64\*\(x^2\+x^3\+x^5\+x^6\)\+ x^72\*\(x^2\+x^5\+x^6\)

r = x^100 \-x^93 \+ x^91 \+ x^88\+ x^81\- x^70 \+ x^66 \+ x^64 \- x^62 \- x^51 \- x^48 \- x^36 \- x^20\+x^17 \- x^16 \- x^15\+ x^8 \- x^3 \+ x^2\+1  \#the ephemeral key


In [0]:
cs = lift(S(h)*S(r))
c = (p*cs + m)%q
a = S(f)*S(c)

In [0]:
lista = [(list(a)[i]%q if list(a)[i]%q<=q/2 else (list(a)[i]%q-q) ) for i in range(len(list(a)))]
b = 0
for i in range(len(list(a))):
    b = b + lista[i]*x^i

In [0]:
nn = S(b)*S(Fp)
nn

In [0]:
mdecrypt = 0
for i in range(N):
    if ZZ(list(nn)[i]%p)<p/2:
        mdecrypt = mdecrypt + ZZ(list(nn)[i]%p)*x^i
    else:
        mdecrypt = mdecrypt + (ZZ(list(nn)[i]%p)-p)*x^i

$$
\textbf{Signing} \text{ --- requires a hash function } H : \mathcal{D} \rightarrow R \text{ on a digital document space } \mathcal{D}. \text{ The properties required of this hash function are explored in appendix D. Signing also requires a norm function } \|\cdot\| : R^2 \rightarrow \mathbb{R} \text{ and a “norm bound” } \mathcal{N} \in \mathbb{R}. \text{ For } (s, t) \in R^2 \text{ we define } \| (s \mod , r \mod q)\| \text{ to be the minimal value of } \|(s + k_1 q, r + k_2 q)\| \text{ for } k_1, k_2 \in \mathbb{R}.
$$



\begin{array}{ll}
1. \quad \textbf{Input:} \text{ A digital document } D \in \mathcal{D} \text{ and the private key } \{f_i, f'_i, h_i\} \text{ for } i = 0 \ldots B. \\
2. \quad  \text{Set } r = 0. \\
3. \quad \text{Set } s = 0. \text{ Set } i = B. \text{ Encode } r \text{ as a bit string. Set } m_0 = H(D \| r), \text{ where} \| \| \text{ denotes concatenation. Set } m = m_0. \\
4. \quad \textbf{Perturb the point using the private lattices:} \text{ While } i \ge 1: \\
    \quad \quad (a) \; x = \lfloor -(1/q)m \ast f'_i \rfloor, \; y = \lfloor (1/q)m \ast f_i \rfloor, \; s_i = x \ast f_i + y \ast f'_i. \\
    \quad \quad (b) \; \text{Set } m = s_i \ast (h_i - h_{i-1}) \mod q. \\
    \quad \quad (c) \; \text{Set } s = s + s_i. \text{ Set } i = i - 1. \\
5. \quad  \textbf{Sign the perturbed point using the public lattice:} \\
   \quad  \quad x = \lfloor -(1/q)m \ast f'_0 \rfloor, \; y = \lfloor (1/q)m \ast f_0 \rfloor, \; s_0 = x \ast f_0 + y \ast f'_0, \; s = s + s_0. \\
6. \quad \textbf{Check the Signature:} \\
   \quad  \quad (a) \; \text{Set } b = \| (s, s \ast h - m_0 \mod q) \|. \\
   \quad  \quad (b) \; \text{If } b \ge \mathcal{N}, \text{ set } r = r + 1 \text{ and go to step 3.} \\
7. \quad  \textbf{Output:} \text{ The triplet } (D, r, s). \\
\end{array}



In [4]:
import hashlib
from sage.all import *

In [5]:
private_key = {
    'q': 251,
    'f': [128, 73, 71],
    'f_prime': [1, 1, 1],
    'h': [1, 1, 1],
    'B': 2,
    'N': 310
}

In [6]:
public_key = {
    'h': [1, 1, 1],
    'q': 251,
    'N': 310,
    'B': 2
}

In [7]:
q = Integer(private_key['q'])

In [8]:
f = [Integer(f_val) for f_val in private_key['f']]

In [9]:
f_prime = [Integer(f_prime_val) for f_prime_val in private_key['f_prime']]


In [10]:
h = [Integer(h_val) for h_val in private_key['h']]


In [11]:
B = Integer(private_key['B'])


In [12]:
N = Integer(private_key['N'])


In [13]:
r = 0

Hash function H, here we use SHA\-256 and interpret the result as a Sage Integer



In [14]:
def H(message):
    return Integer(int(hashlib.sha256(message.encode()).hexdigest(), 16))

Encode an integer value into a bit string of given length



In [15]:
def encode_bit_string(value, length):
    return bin(value)[2:].zfill(length)

In [16]:
D = "secret document"

In [17]:
s = Integer(0)
m0 = H(D + encode_bit_string(r, B))
m0

104205208746268067107294235852433092518383088047250217448903416052241079565016

In [18]:
m = m0
i = B
i

2

In [19]:
i >= 1

True

In [20]:
floor(4.546)

4

In [21]:
x = floor((-1/q) * m * f[i])
x

-29476373788784991094095182253078683541056570722528945971602161512785325295284

In [22]:
y = floor((-1/q) * m * f_prime[i])
y

-415160194208239311184439186663080049874036207359562619318340302996976412610

In [23]:
si = x * f[i] + y * f_prime[i]
si

-2093237699197942606991942379155249611464890557506914726603071807710755072377774

In [24]:
m = (si * (h[i] - h[i-1])) % q
m

0

In [25]:
s += si
s

-2093237699197942606991942379155249611464890557506914726603071807710755072377774

In [26]:
i -= 1
i

1

Since $i \ge 1$, we repeat the step:



In [27]:
x = floor((-1/q) * m * f[i])
y = floor((-1/q) * m * f_prime[i])
si = x * f[i] + y * f_prime[i]
m = (si * (h[i] - h[i-1])) % q
s += si
s

-2093237699197942606991942379155249611464890557506914726603071807710755072377774

In [28]:
x = floor((-1/q) * m * f[0])
y = floor((-1/q) * m * f_prime[0])
s0 = x * f[0] + y * f_prime[0]
s += s0
s

-2093237699197942606991942379155249611464890557506914726603071807710755072377774

In [29]:
v = vector([1,2,4])

In [30]:
norm(v)

sqrt(21)

In [31]:
b = norm(vector([s, s * h[0] - m0]) % q)
b

sqrt(44329)

In [32]:
# check if b < N
b < N

sqrt(44329) < 310

In [33]:
RR(b) < N

True

In [34]:
print(D, r, s)

secret document 0 -2093237699197942606991942379155249611464890557506914726603071807710755072377774


In [35]:
signed_document = (D, r, s)

\begin{array}{ll}
\textbf{Verification} \text{ --- requires the same hash function } H, \text{ norm function } \| \cdot \| \text{ and “norm bound” } \mathcal{N} \in \mathbb{R}. \\
\\
1. \quad  \textbf{Input:} \text{ A signed document } (D, r, s) \text{ and the public key } h. \\
2. \quad  \text{Encode } r \text{ as a bit string. Set } m = H(D \| r). \\
3. \quad  \text{Set } b = \| (s, s \ast h - m \mod q) \|. \\
4. \quad  \textbf{Output:} \text{ valid if } b < \mathcal{N}, \text{ invalid otherwise.} \\
\end{array}



In [36]:
def verify(signed_document, public_key):
    D, r, s = signed_document
    h = [Integer(h_val) for h_val in public_key['h']]
    q = Integer(public_key['q'])
    N = Integer(public_key['N'])
    B = Integer(public_key['B'])
    
    m = H(D + encode_bit_string(r, B))
    b = norm(vector([s, s * h[0] - m]) % q)
    return b < N

In [38]:
is_valid = verify(signed_document, public_key)

sqrt(44329) < 310

In [39]:
print(f"Signed Document: {signed_document}")
print(f"Verification result: {'valid' if is_valid else 'invalid'}")

Signed Document: ('secret document', 0, -2093237699197942606991942379155249611464890557506914726603071807710755072377774)
Verification result: valid


## Comparison to ElGamal and RSA Cryposystem



#### Efficiency:

- Computation: NTRU is generally very efficient due to its reliance on polynomial arithmetic and lattices, which can be performed quickly using fast algorithms like the Fast Fourier Transform \(FFT\).
- Key Sizes: typically uses smaller key sizes compared to RSA for equivalent security levels, resulting in faster operations.
- Signatures: The signing and verification processes are efficient, with relatively small signature sizes.



#### Security:

###### NTRU:

- Security Basis: based on the hardness of lattice problems, specifically the Shortest Vector Problem \(SVP\), which is considered secure against both classical and quantum attacks.
- Quantum Resistance: resistant to quantum attacks, making it a strong candidate for post\-quantum cryptography.

###### ElGamal:

- security based on the Discrete Logarithm Problem \(DLP\) over finite fields.
- not quantum\-resistant; Shor's algorithm can break it if large quantum computers become practical.

###### RSA

- security relies on the hardness of factoring large composite numbers.
- not quantum\-resistant, as Shor's algorithm can efficiently factorize large integers.



#### Ease of Key Generation

###### NTRU:

- Key Generation: NTRU key generation involves generating polynomial rings and finding inverses, which can be done efficiently.
- Complexity: While the mathematics is more complex than RSA, the actual key generation process is quite fast.

###### ElGamal:

- key generation involves choosing a large prime and a generator, which is straightforward but can be computationally expensive due to the need for large primes.
- The process is relatively simple, but finding appropriate parameters can be time\-consuming.

###### RSA:

- key generation requires generating large prime numbers and computing their product, which can be computationally intensive.
- The process is simpler than NTRU in terms of understanding but can be slower due to the need for large primes.



1